<p align="center">
  <img src="https://www.portafolio.co/files/article_new_multimedia/uploads/2023/10/12/652828e536642.jpeg" style="width:100%; max-width:900px; height:180px; object-fit:cover; border-radius:10px;"/>
</p>
<div style="text-align:center;">
  <h1 style="color:#FFD700; display:inline-block; margin:0;">Optimización del Transporte en Nueva York</h1>
  <p>
    <b>Green Taxi | Machine Learning & Data Science | CRISP-DM</b><br>
    <span style="font-size:1.1em;">Análisis y predicción de tarifas y duración de viajes usando datos reales de taxis verdes de NYC.</span>
  </p>
</div>

In [ ]:
# Verifica si las librerías necesarias ya están instaladas
try:
    # Ignorar warnings
    import warnings
    warnings.filterwarnings('ignore')
    # Librerías del sistema
    import sys, subprocess
    # Importaciones base
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    import joblib
    import scipy.stats as stats
    import pickle as pkl

    # Establece el tema global
    sns.set_theme(
    style="whitegrid",                  # El estilo del gráfico (fondo y cuadrícula)
        palette="colorblind",           # La paleta de colores
        font="sans-serif",               # La familia de la fuente
        font_scale=0.9,                  # Escala de la fuente
        rc={"axes.titlesize": 14, "axes.labelsize": 12, "xtick.labelsize": 10, "ytick.labelsize": 10}
    )
    
    print("Dependencias ya instaladas.")

except ImportError:

    print("Dependencias no encontradas. Instalando ahora...")
    
    # Se ejecutan los comandos de instalación
    %pip install --quiet matplotlib
    %pip install --quiet seaborn
    %pip install --quiet joblib
    %pip install --quiet scipy
    
    print("Instalación completada.")

In [ ]:
# Función lambda para verificar e instalar
instalar = lambda p: subprocess.call([sys.executable, "-c", f"import {p}"]) and subprocess.call([sys.executable, "-m", "pip", "install", p, "-q"])

# Actualizar pip
subprocess.call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"])

# Verificar e instalar condicionalmente
if instalar("pyarrow") == 0: print("PyArrow instalado/verificado")
if instalar("fastparquet") == 0: print("FastParquet instalado/verificado")

print("Listo para usar")

# 03.- Data Preparation

In [ ]:
# Configuración del dataset desde el directorio local
url = "cleaned_green_tripdata_2025-06.parquet"
df = pd.read_parquet(url, engine="pyarrow")

In [ ]:
df.head()

In [ ]:
columnas_a_tratar =  df.select_dtypes(include=['float64']).columns.tolist()
columnas_a_tratar

In [ ]:
'''
# ANÁLISIS Y TRATAMIENTO DE OUTLIERS CON MÉTODO IQR
import pandas as pd # Make sure pandas is imported

df_clean = df.copy()

# Identify numerical columns (float64)
columnas_a_tratar = df.select_dtypes(include=['float64']).columns
print("\n--- Detección y Corrección de Outliers (Método IQR) ---")

# Iterate over columns to detect and correct outliers
outliers_found_summary = [] # List to store messages for columns with outliers

for columna in columnas_a_tratar:
    if pd.api.types.is_numeric_dtype(df_clean[columna]):
        
        # Calculate Q1, Q3, and IQR
        Q1 = df_clean[columna].quantile(0.25)
        Q3 = df_clean[columna].quantile(0.75)
        IQR = Q3 - Q1
        
        # Define outlier limits
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        # Detect outliers
        outliers_indices = (df_clean[columna] < limite_inferior) | (df_clean[columna] > limite_superior)
        outliers_count = outliers_indices.sum() # More direct way to count True values
        
        if outliers_count > 0:
            total_rows = len(df_clean)
            percentage = (outliers_count / total_rows) * 100
            
            # --- Simpler print statement ---
            message = (f"Columna '{columna}': {outliers_count} outliers ({percentage:.2f}%) "
                       f"acotados entre {limite_inferior:.2f} y {limite_superior:.2f}.")
            outliers_found_summary.append(message)
            
            # Correct outliers using clip
            df_clean[columna] = df_clean[columna].clip(lower=limite_inferior, upper=limite_superior)
        # else: # You can uncomment this if you want confirmation for columns without outliers
            # print(f"Columna '{columna}': No se encontraron outliers.")

# Print the summary of corrections made
if outliers_found_summary:
    print("Correcciones realizadas:")
    for msg in outliers_found_summary:
        print(f"- {msg}")
else:
    print("No se encontraron outliers en las columnas analizadas.")

print("\n--- Dimensiones del dataset después de la corrección ---")
print(f"Total de columnas: {df_clean.shape[1]}")
print(f"Total de filas: {df_clean.shape[0]}")
'''

In [ ]:
# ANÁLISIS Y TRATAMIENTO DE OUTLIERS CON PERCENTILES 3% Y 97%
df_clean_pct = df.copy()

# Identificar columnas numéricas (puedes ajustar esto si quieres incluir int64 también)
columnas_a_tratar = df.select_dtypes(include=['float64']).columns 

print("\n--- Corrección de Outliers (Acotación por Percentiles 3% y 97%) ---")
print(f"{'Columna':<25} | {'Límite Inferior (P3)':>20} | {'Límite Superior (P97)':>20}")
print("-" * 75)

for columna in columnas_a_tratar:
    if pd.api.types.is_numeric_dtype(df_clean_pct[columna]):
        
        # --- Calcular límites usando percentiles 5 y 95 ---
        limite_inferior = df_clean_pct[columna].quantile(0.03) 
        limite_superior = df_clean_pct[columna].quantile(0.97)
        
        # Contar cuántos valores serán acotados (opcional, para información)
        outliers_bajos = df_clean_pct[df_clean_pct[columna] < limite_inferior].shape[0]
        outliers_altos = df_clean_pct[df_clean_pct[columna] > limite_superior].shape[0]
        total_outliers = outliers_bajos + outliers_altos
        percentage = (total_outliers / len(df_clean_pct)) * 100

        print(f"{columna:<25} | {limite_inferior:>20.2f} | {limite_superior:>20.2f} | {total_outliers} acotados ({percentage:.2f}%)")
        
        # --- Aplicar la acotación ---
        df_clean_pct[columna] = df_clean_pct[columna].clip(lower=limite_inferior, upper=limite_superior)

print("\n--- Dimensiones del dataset después de la corrección ---")
print(f"Total de columnas: {df_clean_pct.shape[1]}")
print(f"Total de filas: {df_clean_pct.shape[0]}")

# Puedes verificar con df_clean_pct.describe()


'''
Acotación por Percentiles (Winsorizing) 

Esta es una alternativa muy directa y controlable. En lugar de usar los límites calculados por el IQR, defines los límites directamente como percentiles de tus datos.

Concepto: Reemplazas los valores extremos con los valores de los percentiles más cercanos. Por ejemplo, reemplazas todos 
los valores por debajo del percentil 5 con el valor del percentil 5, y todos los valores por encima del percentil 95 con el valor del percentil 95. 
Los percentiles comunes a usar son (1% y 99%) o (5% y 95%).

Ventajas: Tienes control directo sobre qué porcentaje de tus datos se ve afectado. Es robusto a la forma de la distribución.

Implementación: Es muy fácil con .quantile() y .clip().
'''


In [ ]:
# 4. Verificar los resultados
# Compara las estadísticas del dataset original y el limpio.
# --- Estadísticas del DataFrame ORIGINAL ---
print("     Estadísticas del DataFrame ORIGINAL")
stats_original = df[columnas_a_tratar].describe().round(2).T
stats_original

In [ ]:
# --- Estadísticas del DataFrame LIMPIO ---
print("       Estadísticas del DataFrame LIMPIO")
stats_clean = df_clean_pct[columnas_a_tratar].describe().round(2).T
stats_clean

In [ ]:
df_clean_pct = df_clean_pct.copy()
print("Limpieza de outliers completada. DataFrame actualizado.")

In [ ]:
'''
# Procesamiento de valores faltantes
# Seleccionar columnas numéricas
cols_corr = df_clean_pct.select_dtypes(include=[np.number]).columns.tolist()

# Validar que las columnas existan y sean numéricas
cols_corr = [col for col in cols_corr if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

# Crear copia de trabajo
df_corr = df[cols_corr].copy()

# --- Inicio de la Impresión Mejorada ---
print("--- Preprocesamiento para Matriz de Correlación ---")
print(f"Columnas numéricas seleccionadas: {len(cols_corr)}")

print("\nRellenando valores faltantes (NaN) con la mediana:")
imputed_cols_count = 0
for col in df_corr.columns:
    missing_before = df_corr[col].isnull().sum()
    if missing_before > 0:
        median_val = df_corr[col].median()
        df_corr[col] = df_corr[col].fillna(median_val)
        # Imprime de forma más clara y con indentación
        print(f"  - Columna '{col}': {missing_before} valores NaN rellenados con mediana {median_val:.2f}")
        imputed_cols_count += 1

if imputed_cols_count == 0:
    print("  (No se encontraron valores NaN en las columnas numéricas seleccionadas)")

# Actualiza el mensaje si no hay filtrado posterior, o elimínalo
print(f"\nDataFrame listo para correlación. Dimensiones: {df_corr.shape}") 
'''

In [ ]:
'''' ANÁLISIS Y TRATAMIENTO DE VALORES NULOS CON ELIMINACIÓN DE FILAS
# Crear una copia para no modificar df_clean original
df_deleted = df_clean.copy()

# Definir las columnas donde buscaremos nulos para eliminar la fila
columnas_con_nulos = ['passenger_count', 'trip_type', 'store_and_fwd_flag', 
                      'RatecodeID', 'congestion_surcharge', 'payment_type']

# Guardar el número de filas antes de eliminar
rows_before = len(df_deleted)

# Eliminar las filas que tengan al menos un valor nulo en las columnas especificadas
df_deleted.dropna(subset=columnas_con_nulos, inplace=True)

# Guardar el número de filas después de eliminar
rows_after = len(df_deleted)

print("--- Estrategia: Eliminación de Filas (Listwise Deletion) ---")
print(f"Filas originales: {rows_before}")
print(f"Filas después de eliminar nulos: {rows_after}")
print(f"Total de filas eliminadas: {rows_before - rows_after}")

# Ahora puedes trabajar con 'df_deleted'
# print("\nVerificación de nulos restantes en columnas clave:")
# print(df_deleted[columnas_con_nulos].isnull().sum())
Ventajas: Simple y rápido. Asegura que no queden nulos en esas columnas. Desventajas: Se pierde información 
(en tu caso, alrededor del 7.7% de las filas). Puede introducir sesgo si los datos faltantes no son aleatorios.
'''

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer
import numpy as np # Necesario para np.number

# Crear una copia para no modificar df_clean original
df_knn = df_clean_pct.copy()

# --- Paso A: Imputar columnas categóricas con nulos (usando la moda) ---
col_categoricas_nulos = ['trip_type', 'store_and_fwd_flag', 'RatecodeID', 'payment_type']
print("--- Estrategia: Imputación KNN (y Moda para Categóricas) ---")
print("Imputando categóricas con la moda:")
for col in col_categoricas_nulos:
    if col in df_knn.columns:
        mode_val = df_knn[col].mode()[0]
        df_knn[col].fillna(mode_val, inplace=True)
        print(f"- Columna '{col}' imputada con moda: {mode_val}")

# --- Paso B: Imputar columnas numéricas con nulos usando KNN ---
# Seleccionar solo las columnas numéricas que tienen nulos
cols_numericas_nulos = ['passenger_count', 'congestion_surcharge'] 
# Asegúrate de que solo incluyes columnas numéricas aquí

# Verificar que las columnas existan y tengan nulos
cols_a_imputar_knn = [col for col in cols_numericas_nulos if col in df_knn.columns and df_knn[col].isnull().any()]

if cols_a_imputar_knn:
    print("\nImputando numéricas con KNN:")
    
    # Crear el imputador KNN (puedes ajustar n_neighbors)
    imputer = KNNImputer(n_neighbors=5) 
    
    # Aplicar el imputador SOLO a las columnas numéricas con nulos
    # .fit_transform devuelve un array NumPy, lo reconvertimos a DataFrame
    df_knn[cols_a_imputar_knn] = imputer.fit_transform(df_knn[cols_a_imputar_knn])
    
    # KNN puede devolver floats, así que redondeamos y convertimos a entero si es apropiado
    if 'passenger_count' in cols_a_imputar_knn:
         # Usamos Int64 (con mayúscula) para permitir posibles NaN si algo falla, aunque no debería
         df_knn['passenger_count'] = df_knn['passenger_count'].round().astype('Int64') 
    
    print(f"- Columnas imputadas con KNN: {cols_a_imputar_knn}")
else:
    print("\nNo se encontraron columnas numéricas con nulos para imputar con KNN.")


# Ahora puedes trabajar con 'df_knn'
print("\nVerificación de nulos restantes en todo el DataFrame:")
print(df_knn.isnull().sum().sort_values(ascending=False).head())

'''
Imputación KNN (K-Nearest Neighbors Imputation) 
Este método es más sofisticado. Utiliza los valores de las filas "vecinas" (las más similares en otras características) para estimar y 
rellenar los valores faltantes. Solo funciona con datos numéricos. Tendrás que manejar las columnas categóricas (store_and_fwd_flag) por separado 
(por ejemplo, con imputación por moda, como vimos antes).
Ventajas: Puede ser más preciso que la imputación simple (media/mediana) porque considera otras características. Conserva todas las filas. 
Desventajas: Computacionalmente más costoso que la imputación simple o la eliminación. 
Sensible a la escala de los datos (aunque KNNImputer suele manejar esto). Requiere manejar categóricas por separado. 
La elección de k (n_neighbors) puede influir en el resultado.
'''

In [ ]:
# Mapa de calor de valores nulos
plt.figure(figsize=(12, 6))
sns.heatmap(df_knn.isnull(), cbar=False)
plt.title("Mapa de calor de valores nulos")
ax = plt.gca()
ax.tick_params(axis='x', rotation=75)

## Características Temporales

In [ ]:
# dataframe DataPreparation
df_prep =  df_knn
df_prep.head()

In [ ]:
# Crear una columna de fecha a partir de año, mes y día
df_prep['pickup_date'] = pd.to_datetime(df_prep[['pickup_year', 'pickup_month', 'pickup_day']].rename(
	columns={'pickup_year': 'year', 'pickup_month': 'month', 'pickup_day': 'day'}
))

# Extraer el día de la semana usando los datos disponibles
df_prep['pickup_day_of_week'] = df_prep['pickup_date'].dt.dayofweek
df_prep['pickup_weekend'] = df_prep['pickup_day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
# Es fin de semana si es sábado (5) o domingo (6)
df_prep['pickup_is_weekend'] = df_prep['pickup_day_of_week'].isin([5, 6]).astype(int)

In [ ]:
#Categorizar el día en Manaña, Tarde, Noche
df_prep['pickup_time_of_day'] = pd.cut(
	df_prep['pickup_hour'],
	bins=[0, 5, 12, 17, 21, 24],
	labels=['Night', 'Morning', 'Afternoon', 'Evening', 'Late Night'],
	right=False,
	ordered=False
)

In [ ]:
# hora Punta 
df_prep['is_peak_hour'] = df_prep['pickup_hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)

## Caracteristicas del viaje

In [ ]:
#Calcular promedio de velocidad del viaje (millas por hora)
df_prep['average_speed_mph'] = np.where(
    df_prep['trip_duration_min'] > 0,
    (df_prep['trip_distance'] / (df_prep['trip_duration_min'] / 60)).round(2),
    0
)

#Costo por milla: Calcular total_amount / trip_distance o fare_amount / trip_distance. (¡Ojo con divisiones por cero!).
df_prep['cost_per_mile_total'] = np.where(
    df_prep['trip_distance'] > 0,
    (df_prep['total_amount'] / df_prep['trip_distance']).round(2),
    0
)

#Costo por minuto: Calcular total_amount / trip_duration_min o fare_amount / trip_duration_min. (¡Ojo con divisiones por cero!).
df_prep['cost_per_minute_total'] = np.where(
    df_prep['trip_duration_min'] > 0,
    (df_prep['total_amount'] / df_prep['trip_duration_min']).round(2),
    0
)

#Porcentaje de propina: Calcular (tip_amount / fare_amount) * 100 o (tip_amount / total_amount) * 100. (Manejar casos donde fare_amount o total_amount sean cero).
df_prep['tip_percentage_fare'] = np.where(
    df_prep['fare_amount'] > 0,
    (df_prep['tip_amount'] / df_prep['fare_amount'] * 100).round(2),
    0
)

#Ratio de peajes sobre tarifa: tolls_amount / fare_amount.
df_prep['tolls_to_fare_ratio'] = np.where(
    df_prep['fare_amount'] > 0,
    (df_prep['tolls_amount'] / df_prep['fare_amount']).round(2),
    0
)

#Suma de recargos: extra + mta_tax + improvement_surcharge + congestion_surcharge + cbd_congestion_fee.
df_prep['surcharges_total'] = (df_prep['extra'] + df_prep['mta_tax'] + df_prep['improvement_surcharge'] + df_prep['congestion_surcharge'] + df_prep['cbd_congestion_fee']).round(2)
#Ratio de recargos sobre total: Suma de recargos / total_amount.
df_prep['surcharges_to_total_ratio'] = np.where(
    df_prep['total_amount'] > 0,
    (df_prep['surcharges_total'] / df_prep['total_amount']).round(2),
    0
)

In [ ]:
df_prep.sample(10)

In [ ]:
print("\n--- Eliminando Columnas con Varianza Cero")
print("---") # Separador Markdown
df_prep = df_knn
columnas_varianza_cero_info = [] # Para guardar info de columnas a eliminar
for col in df_prep.columns:
    # Usar una tolerancia pequeña por si acaso la varianza es casi cero
    if pd.api.types.is_numeric_dtype(df_prep[col]) and df_prep[col].var() < 1e-10: 
        unique_val = df_prep[col].unique()[0]
        columnas_varianza_cero_info.append({'col': col, 'val': unique_val})

if columnas_varianza_cero_info:
    print("Se encontraron columnas con varianza cero:")
    for info in columnas_varianza_cero_info:
        # Imprime cada columna como un bullet point
        print(f"- ✗ Columna '{info['col']}': Varianza cero (valor único: {info['val']})")
        
    # Extraer solo los nombres para eliminar
    columnas_a_eliminar = [info['col'] for info in columnas_varianza_cero_info]
    df_prep = df_prep.drop(columns=columnas_a_eliminar)
    print(f"\n✓ Se eliminaron {len(columnas_a_eliminar)} columnas.")

else:
    print("✓ No se encontraron columnas con varianza cero.")

print(f"✓ Dimensiones finales: {df_prep.shape}")
print("---")

In [ ]:
# Seleccionar columnas numéricas y rellenar valores faltantes
numeric_cols_for_corr = df_prep.select_dtypes(include=[np.number]).columns
df_for_corr = df_prep[numeric_cols_for_corr].fillna(df_prep[numeric_cols_for_corr].median())

# Calcular la matriz de correlación
corr = df_for_corr.corr()

# Crear una máscara para valores débiles (opcional)
#mask = np.abs(corr) < 0.2

plt.figure(figsize=(18, 14))
#sns.heatmap(corr, annot=True,cmap='Blues',fmt=".3f",mask=mask,linewidths=.5,cbar_kws={"shrink": .8},vmin=-1,vmax=1)
sns.heatmap(corr, annot=True,cmap='Blues',fmt=".3f",linewidths=.5,cbar_kws={"shrink": .8},vmin=-1,vmax=1)
plt.title("Matriz de Correlación", fontsize=18, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Seleccionar columnas numéricas y rellenar valores faltantes
numeric_cols_for_corr = df_prep.select_dtypes(include=['float64', 'int32', 'Int32']).columns
df_for_corr = df_prep[numeric_cols_for_corr].fillna(df_prep[numeric_cols_for_corr].median())

# Calcular la matriz de correlación
corr = df_for_corr.corr()

# Crear una máscara para valores débiles (opcional)
mask = np.abs(corr) < 0.3

plt.figure(figsize=(18, 14))
sns.heatmap(corr, annot=True,cmap='Blues',fmt=".3f",mask=mask,linewidths=.5,cbar_kws={"shrink": .8},vmin=-1,vmax=1)
plt.title("Matriz de Correlación", fontsize=18, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Define el nombre del archivo
dataunderstanding = 'preparation_green_tripdata_2025-06.parquet'

# Guarda el DataFrame df_prep en formato Parquet
try:
    df_prep.to_parquet(dataunderstanding, index=False) 
    print(f"DataFrame guardado exitosamente como '{dataunderstanding}'")
except Exception as e:
    print(f"Error al guardar como Parquet: {e}")
    # Podrías necesitar instalar pyarrow: pip install pyarrow